In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df=pd.read_csv("qoute_dataset.csv")

In [3]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [4]:
quotes=df["quote"]
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: object

In [5]:
quotes=quotes.str.lower()
quotes[0]


'“the world as we have created it is a process of our thinking. it cannot be changed without changing our thinking.”'

In [6]:
import string
translator=str.maketrans("","",string.punctuation)
quotes=df["quote"].apply(lambda x : x.translate(translator))

In [7]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
vocab_size=85280

tokenize=Tokenizer(num_words=vocab_size)
tokenize.fit_on_texts(quotes)

In [9]:
word_index=tokenize.word_index
print(len(word_index))

8978


In [10]:
seq=tokenize.texts_to_sequences(quotes)

In [11]:
quotes[0]

'“The world as we have created it is a process of our thinking It cannot be changed without changing our thinking”'

In [12]:
seq[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [13]:
x=[]
y=[]

In [14]:
for seq in seq:
    for i in range(1,len(seq)):
        input_seq=seq[:i]
        output_seq=seq[i]
        x.append(input_seq)
        y.append(output_seq)

In [15]:
x

[[713],
 [713, 62],
 [713, 62, 29],
 [713, 62, 29, 19],
 [713, 62, 29, 19, 16],
 [713, 62, 29, 19, 16, 946],
 [713, 62, 29, 19, 16, 946, 10],
 [713, 62, 29, 19, 16, 946, 10, 7],
 [713, 62, 29, 19, 16, 946, 10, 7, 5],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104],
 [713,
  62,
  29,
  19,
  16,
  946,
  10,
  7,
  5,
  1156,
  8,
  70,
  293,
  10,
  145,
  12,
  809,
  104,
  752],
 [713,
  62,
  29,
  19,
  16,
  946,
  10,
  7,
  5,
  1156,
  8,
  70,
  293,
  10,
  145,
  12,
  809,
  

In [16]:
max_len=max(len(x) for x in x)
print(max_len)


745


In [17]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded=pad_sequences(x,maxlen=max_len,padding="pre")

In [18]:
X_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]],
      shape=(85271, 745), dtype=int32)

In [19]:
y=np.array(y)

In [20]:
from tensorflow.keras.utils import to_categorical
y_one_hot=to_categorical(y,num_classes=vocab_size)

In [21]:
y.shape

(85271,)

In [22]:
y_one_hot.shape

(85271, 10000)

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,Dense,Embedding,SimpleRNN

In [24]:
Embed_dim=50
rnn_units=128

In [25]:
rnn_model=Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size,output_dim=Embed_dim,input_length=max_len)
)

rnn_model.add(SimpleRNN( units=rnn_units))
rnn_model.add(Dense(units=vocab_size,activation="softmax"))

c:\Users\mansi\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [26]:
rnn_model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])

In [27]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
lstm_model=Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size,output_dim=Embed_dim,input_length=max_len)
)
lstm_model.add(
    Dense(units=vocab_size,activation="softmax"))
lstm_model.add(LSTM(units=rnn_units))

In [38]:
lstm_model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])

In [39]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 745, 50)        │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 745, 10000)     │       510,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │     5,186,048 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,196,048 (23.64 MB)

 Trainable params: 6,196,048 (23.64 MB)

 Non-trainable params: 0 (0.00 B)

In [31]:
epochs=10
batch_size=128


In [33]:
histor_rnn=rnn_model.fit(
    X_padded,y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    )

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 170s 279ms/step - accuracy: 0.0442 - loss: 6.7071 - val_accuracy: 0.0595 - val_loss: 6.5703
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 278s 463ms/step - accuracy: 0.0767 - loss: 6.1336 - val_accuracy: 0.0863 - val_loss: 6.3502
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 232s 386ms/step - accuracy: 0.0984 - loss: 5.8066 - val_accuracy: 0.0996 - val_loss: 6.3362
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 250s 417ms/step - accuracy: 0.1116 - loss: 5.5399 - val_accuracy: 0.1013 - val_loss: 6.3571
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 283s 472ms/step - accuracy: 0.1261 - loss: 5.2985 - val_accuracy: 0.1071 - val_loss: 6.3716
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 182s 304ms/step - accuracy: 0.1395 - loss: 5.0659 - val_accuracy: 0.1083 - val_loss: 6.4145
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 629s 1s/step - accuracy: 0.1523 - loss: 4.8468 - val_accuracy: 0.1106 - val_loss: 6.4862
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 265s 441ms/step - accuracy: 0.1691 - lo

In [ ]:
lstm_model.summary()

In [40]:
echos=100
batch_size=128

histor_lstm=lstm_model.fit(
    X_padded,y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    )

MemoryError: Unable to allocate 2.86 GiB for an array with shape (76743, 10000) and data type float32